# Data Loading and Overview

Load the CDC Diabetes Health Indicators dataset, inspect its structure, and review basic quality checks before any analysis or modelling.

In [1]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append(str(Path.cwd().parent / 'src'))
from data_loader import load_cdc_diabetes_data

## Load dataset

In [2]:
X, y, df = load_cdc_diabetes_data()

print(f"Dataset shape:  {df.shape}")
print(f"Features:       {X.shape[1]}")
print(f"Samples:        {X.shape[0]:,}")
print(f"Target column:  {y.name}")

Dataset shape:  (253680, 22)
Features:       21
Samples:        253,680
Target column:  Diabetes_binary


## Column inventory

In [3]:
print("Columns and dtypes:")
print(df.dtypes.to_string())

Columns and dtypes:
Diabetes_binary         int64
HighBP                  int64
HighChol                int64
CholCheck               int64
BMI                     int64
Smoker                  int64
Stroke                  int64
HeartDiseaseorAttack    int64
PhysActivity            int64
Fruits                  int64
Veggies                 int64
HvyAlcoholConsump       int64
AnyHealthcare           int64
NoDocbcCost             int64
GenHlth                 int64
MentHlth                int64
PhysHlth                int64
DiffWalk                int64
Sex                     int64
Age                     int64
Education               int64
Income                  int64


## Descriptive statistics

In [4]:
df.describe(include='number').round(2)

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,253680.00,253680.00,253680.00,253680.00,253680.00,253680.00,253680.00,253680.00,253680.00,253680.00,...,253680.00,253680.00,253680.00,253680.00,253680.00,253680.00,253680.00,253680.00,253680.00,253680.00
mean,0.14,0.43,0.42,0.96,28.38,0.44,0.04,0.09,0.76,0.63,...,0.95,0.08,2.51,3.18,4.24,0.17,0.44,8.03,5.05,6.05
std,0.35,0.49,0.49,0.19,6.61,0.50,0.20,0.29,0.43,0.48,...,0.22,0.28,1.07,7.41,8.72,0.37,0.50,3.05,0.99,2.07
min,0.00,0.00,0.00,0.00,12.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00
25%,0.00,0.00,0.00,1.00,24.00,0.00,0.00,0.00,1.00,0.00,...,1.00,0.00,2.00,0.00,0.00,0.00,0.00,6.00,4.00,5.00
50%,0.00,0.00,0.00,1.00,27.00,0.00,0.00,0.00,1.00,1.00,...,1.00,0.00,2.00,0.00,0.00,0.00,0.00,8.00,5.00,7.00
75%,0.00,1.00,1.00,1.00,31.00,1.00,0.00,0.00,1.00,1.00,...,1.00,0.00,3.00,2.00,3.00,0.00,1.00,10.00,6.00,8.00
max,1.00,1.00,1.00,1.00,98.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,5.00,30.00,30.00,1.00,1.00,13.00,6.00,8.00


## Missing values

In [5]:
missing = df.isna().sum().sort_values(ascending=False)
cols_with_missing = missing[missing > 0]

if cols_with_missing.empty:
    print("No missing values detected.")
else:
    print(f"{len(cols_with_missing)} column(s) with missing values:")
    print(cols_with_missing)

No missing values detected.


## Target distribution

In [6]:
counts = y.value_counts()
pct = y.value_counts(normalize=True).mul(100).round(1)

print(f"Target: {y.name}")
print()
for label in counts.index:
    print(f"  class {label}: {counts[label]:>6,}  ({pct[label]}%)")

print(f"\nClass imbalance ratio: {counts.max() / counts.min():.1f}:1")

Target: Diabetes_binary

  class 0: 218,334  (86.1%)
  class 1: 35,346  (13.9%)

Class imbalance ratio: 6.2:1


The dataset is imbalanced: the positive class (diabetes = 1) represents roughly 14–15% of the samples. This will need to be accounted for when selecting evaluation metrics — accuracy alone would be misleading here.

## Quick sanity check on feature ranges

In [7]:
# Binary features should only contain 0 and 1
binary_candidates = [c for c in X.columns if X[c].nunique() <= 2]
print(f"Binary features ({len(binary_candidates)}): {binary_candidates}")

# Continuous/ordinal features
continuous = [c for c in X.columns if c not in binary_candidates]
print(f"\nContinuous / ordinal features ({len(continuous)}):")
print(X[continuous].agg(['min', 'max']).T.to_string())

Binary features (14): ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex']

Continuous / ordinal features (7):
           min  max
BMI         12   98
GenHlth      1    5
MentHlth     0   30
PhysHlth     0   30
Age          1   13
Education    1    6
Income       1    8
